![](https://media.licdn.com/dms/image/v2/D4E0BAQGumdWW8R2giQ/company-logo_200_200/B4EZ87y9pLK4AE-/0/1783414643504/health_data_research_service_logo?e=2147483647&v=beta&t=GxsLjCFo9AmdCTHkDLK622NOIF3hplTcipdZBeJbzPY)

# Technology Feasibility - Demo 1

## Objective of the demo

Demonstrate that we can access synthetic data held by different 'organisations' in different clouds through a single interface and query engine

---

## Scene Setting

Oxford University Hospitals, University Hospitals Sussex and University Hospital Birmingham are collaborating in order to analyse medication usage across a rare cancer, in this case penile cancer.

---

#### Demo

1. Look at the available data
2. Show where the data is stored
3. Answer a research question
4. Upcoming demos



#### Look at the available data


In [ ]:
from pyspark.sql import DataFrame
import pandas as pd

In [25]:
import os
## Import libraries and connect to the query engine

from pyspark.sql import SparkSession
from pyspark.sql import functions as f
# The Spark Connect server has no catalogs of its own: each session registers the Polaris catalogs
# it uses, authenticated with the user's OIDC token, which JupyterHub injects into the pod.
OIDC_TOKEN = os.environ["OIDC_TOKEN"]
POLARIS_DOMAIN_NAME = os.environ.get("POLARIS_DOMAIN_NAME", "emryspolaris.click")
OIDC_DOMAIN_NAME = os.environ.get("OIDC_DOMAIN_NAME", "emrysoauth.click")


def with_catalog(builder, name, warehouse=None):
    """Registers the Polaris catalog warehouse (default: name) as the Spark catalog name."""
    prefix = f"spark.sql.catalog.{name}"
    return (builder
            .config(prefix, "org.apache.iceberg.spark.SparkCatalog")
            .config(f"{prefix}.type", "rest")
            .config(f"{prefix}.uri", f"https://{POLARIS_DOMAIN_NAME}/api/catalog")
            .config(f"{prefix}.warehouse", warehouse or name)
            .config(f"{prefix}.cache-enabled", "false")
            .config(f"{prefix}.header.X-Iceberg-Access-Delegation", "vended-credentials")
            .config(f"{prefix}.oauth2-server-uri", f"https://{OIDC_DOMAIN_NAME}/realms/polaris/protocol/openid-connect/token")
            .config(f"{prefix}.scope", "openid")
            .config(f"{prefix}.token", OIDC_TOKEN))


builder = SparkSession.builder.appName("test").remote("sc://spark-connector-0-driver-svc:15002")
for catalog in ["aws", "azure", "gcp"]:
    builder = with_catalog(builder, catalog)
spark = builder.getOrCreate()


In [ ]:
spark.sql(" tables in aws.acceptancetests").filter(f.col("tableName").startswith("ouh")).select(f.col("tableName")).show()


In [26]:
spark.sql("show tables in aws.acceptancetests").filter(f.col("tableName").startswith("ouh")).select(f.col("tableName")).show()
spark.sql("show tables in azure.acceptancetests").filter(f.col("tableName").startswith("uhs")).select(f.col("tableName")).show()
spark.sql("show tables in gcp.acceptancetests").filter(f.col("tableName").startswith("uhb")).select(f.col("tableName")).show()

+---------------+
|      tableName|
+---------------+
| ouh_conditions|
|ouh_medications|
|   ouh_patients|
+---------------+

+---------------+
|      tableName|
+---------------+
| uhs_conditions|
| uhs_medication|
|uhs_medications|
|   uhs_patients|
+---------------+

+---------------+
|      tableName|
+---------------+
| uhb_conditions|
|uhb_medications|
|   uhb_patients|
+---------------+



### Show where the data is stored

In [27]:
spark.sql("describe table extended aws.acceptancetests.ouh_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("describe table extended azure.acceptancetests.uhs_conditions").filter("col_name = 'Location'").show(truncate=False)
spark.sql("describe table extended gcp.acceptancetests.uhb_conditions").filter("col_name = 'Location'").show(truncate=False)


+--------+-----------------------------------------------------+-------+
|col_name|data_type                                            |comment|
+--------+-----------------------------------------------------+-------+
|Location|s3://emrys-afon-bucket/acceptancetests/ouh_conditions|       |
+--------+-----------------------------------------------------+-------+

+--------+------------------------------------------------------------------------------------------+-------+
|col_name|data_type                                                                                 |comment|
+--------+------------------------------------------------------------------------------------------+-------+
|Location|abfss://afonstorage@afonstorageaccount.dfs.core.windows.net/acceptancetests/uhs_conditions|       |
+--------+------------------------------------------------------------------------------------------+-------+

+--------+---------------------------------------------------+-------+
|col_name|da


### Research Question

**What combination of medications are patients diagnosed with stage 4 penile cancer prescribed?**


In [28]:
spark.sql(""" describe table gcp.acceptancetests.uhb_conditions""").show()
spark.sql(""" describe table gcp.acceptancetests.uhb_medications""").show()


+-------------------+---------+-------+
|           col_name|data_type|comment|
+-------------------+---------+-------+
|       condition_id|   string|   NULL|
|         patient_id|   string|   NULL|
|        snomed_code|   bigint|   NULL|
|         icd10_code|   string|   NULL|
|        description|   string|   NULL|
|    clinical_status|   string|   NULL|
|verification_status|   string|   NULL|
|              stage|   string|   NULL|
|     onset_datetime|   string|   NULL|
|      recorded_date|   string|   NULL|
+-------------------+---------+-------+

+--------------------+---------+-------+
|            col_name|data_type|comment|
+--------------------+---------+-------+
|medication_reques...|   string|   NULL|
|          patient_id|   string|   NULL|
|        condition_id|   string|   NULL|
|         rxnorm_code|   bigint|   NULL|
|          medication|   string|   NULL|
|              status|   string|   NULL|
|              intent|   string|   NULL|
|         authored_on|   stri

In [29]:
spark.sql("""
select count(distinct patient_id) as uhb_penile_cancer_patients from gcp.acceptancetests.uhb_conditions 
where snomed_code = "363515000" and stage = "Stage 4" """).show()

spark.sql("""
select count(distinct patient_id) as ouh_penile_cancer_patients from aws.acceptancetests.ouh_conditions 
where snomed_code = "363515000" and stage = "Stage 4" """).show()

spark.sql("""
select count(distinct patient_id) as uhs_penile_cancer_patients from azure.acceptancetests.uhs_conditions 
where snomed_code = "363515000" and stage = "Stage 4" """).show()

+--------------------------+
|uhb_penile_cancer_patients|
+--------------------------+
|                         4|
+--------------------------+

+--------------------------+
|ouh_penile_cancer_patients|
+--------------------------+
|                         3|
+--------------------------+

+--------------------------+
|uhs_penile_cancer_patients|
+--------------------------+
|                         6|
+--------------------------+



One query that accesses and joins data from across three clouds and returns the row level data needed

In [31]:


def sql_generator(cloud,table1,table2):
    sql = f"""select B.patient_id,B.medication from (
select patient_id from {cloud}.acceptancetests.{table1} 
where snomed_code = "363515000" and stage = "Stage 4") A
left join 
(select patient_id,medication from {cloud}.acceptancetests.{table2}) B
on A.patient_id = B.patient_id
"""
    return sql

unioned_data = sql_generator('azure','uhs_conditions','uhs_medication') + " UNION All \n" + \
sql_generator('gcp','uhb_conditions','uhb_medications') + " UNION All \n" + \
sql_generator('aws','ouh_conditions','ouh_medications') 

all = spark.sql(unioned_data).toPandas()


In [32]:
pd.set_option("display.max_colwidth",200)
all.groupby('patient_id').agg(lambda x : " : ".join(set(x))).groupby('medication').size().sort_values().reset_index()
        

,medication,0
0,cisplatin 1 MG/ML Injectable Solution : fluorouracil : paclitaxel 6 MG/ML Injectable Solution,1
1,cisplatin 1 MG/ML Injectable Solution : ifosfamide,1
2,cisplatin 1 MG/ML Injectable Solution : paclitaxel 6 MG/ML Injectable Solution,1
3,fluorouracil : ifosfamide,1
4,fluorouracil : paclitaxel 6 MG/ML Injectable Solution : ifosfamide,1
5,fluorouracil : paclitaxel 6 MG/ML Injectable Solution,2
6,cisplatin 1 MG/ML Injectable Solution : fluorouracil : paclitaxel 6 MG/ML Injectable Solution : ifosfamide,6
